## 1️.Install Libraries

In [1]:
# !pip install pdfplumber pytesseract pillow pandas openpyxl google-generativeai reportlab

## 2️.Configuration

In [2]:
import os
import pytesseract
os.environ["GEMINI_API_KEY"] = ""  # ← Paste your key
EXCEL_FILE     = "gst_data1.xlsx"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
print("Config loaded ✔")

Config loaded ✔


## 3️.Text Extraction

In [3]:
import pdfplumber
from PIL import Image, ImageEnhance, ImageFilter
import pytesseract, os

SUPPORTED_EXTENSIONS = {".pdf",".png",".jpg",".jpeg",".tiff",".tif",".bmp",".webp"}

def preprocess_image(image):
    image = image.convert("L")
    image = ImageEnhance.Contrast(image).enhance(2.0)
    image = image.filter(ImageFilter.SHARPEN)
    return image

def table_to_text(table):
    lines = []
    for row in table:
        cells = [str(c).strip() if c else "" for c in row]
        non_empty = [c for c in cells if c]
        if not non_empty: continue
        if len(cells) == 2:
            for c in cells:
                if c: lines.append(c)
        elif len(cells) == 4 and any(":" in c for c in cells):
            for i in range(0, len(cells), 2):
                k = cells[i].strip()
                v = cells[i+1].strip() if i+1 < len(cells) else ""
                if k or v: lines.append(f"{k} {v}".strip())
        else:
            lines.append("  |  ".join(non_empty))
    return "\n".join(lines)

def extract_text_from_pdf(file_path):
    parts = []
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            raw_text   = page.extract_text() or ""
            tables     = page.extract_tables()
            table_texts = [table_to_text(t) for t in tables if t]
            if not raw_text.strip() and not table_texts:
                pil = page.to_image(resolution=300).original
                parts.append(pytesseract.image_to_string(preprocess_image(pil), config="--psm 6 --oem 3"))
                continue
            page_text = raw_text + "\n"
            if table_texts:
                page_text += "\n--- STRUCTURED TABLE DATA ---\n" + "\n\n".join(table_texts)
            parts.append(page_text)
    return "\n\n".join(parts).strip()

def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Unsupported: '{ext}'")
    if ext == ".pdf":
        return extract_text_from_pdf(file_path)
    return pytesseract.image_to_string(
        preprocess_image(Image.open(file_path)), config="--psm 6 --oem 3"
    ).strip()

# text = extract_text("invoice_3_it_services.jpg")
# print(text[:400])

## 4️.OCR Correction Layer

In [4]:
import re

# OCR commonly confuses these character pairs:
# Letter mistaken as digit: O->0, I->1, S->5, B->8, G->6, Z->2
# Digit mistaken as letter: 0->O, 1->I, 5->S, 8->B
TO_DIGIT  = {'O':'0', 'I':'1', 'S':'5', 'B':'8', 'G':'6'}
TO_LETTER = {'0':'O', '1':'I', '5':'S', '8':'B'}

def correct_gstin_ocr(gstin):
    """
    Fix OCR misreads using GSTIN positional rules:
      Pos  0-1  : State code    → must be DIGITS   (e.g. O→0)
      Pos  2-6  : PAN letters   → must be LETTERS  (e.g. 8→B)
      Pos  7-10 : PAN digits    → must be DIGITS   (e.g. O→0)
      Pos  11   : PAN letter    → must be LETTER
      Pos  12   : Entity number → digit or letter (leave as-is)
      Pos  13   : ALWAYS 'Z'    → force Z if anything else (Z→2 is most common OCR error)
      Pos  14   : Check digit   → digit or letter (leave as-is)
    """
    if not gstin:
        return gstin
    g = list(str(gstin).strip().upper())
    if len(g) != 15:
        return ''.join(g)

    # Positions that must be digits: 0, 1, 7, 8, 9, 10
    for i in [0, 1, 7, 8, 9, 10]:
        if g[i] in TO_DIGIT:
            g[i] = TO_DIGIT[g[i]]

    # Positions that must be letters: 2, 3, 4, 5, 6, 11
    for i in [2, 3, 4, 5, 6, 11]:
        if g[i] in TO_LETTER:
            g[i] = TO_LETTER[g[i]]

    # Position 13 is ALWAYS 'Z' — the most common OCR error (Z→2)
    if g[13] != 'Z':
        g[13] = 'Z'

    return ''.join(g)


# ── Verify on real OCR errors ──
tests = [
    ('29AACFN8812L124',  '29AACFN8812L1Z4'),  # Invoice 3 JPG: Z→2
    ('O8AAFCR3201K1ZP',  '08AAFCR3201K1ZP'),  # Invoice 5 JPG: O→0
    ('33AABCS1429B1Z8',  '33AABCS1429B1Z8'),  # No change needed
    ('27AADCM7832H1ZQ',  '27AADCM7832H1ZQ'),  # No change needed
]
print("OCR Correction Results:")
for raw, expected in tests:
    fixed = correct_gstin_ocr(raw)
    status = '✅' if fixed == expected else '❌'
    print(f"  {raw} → {fixed}  {status}")

OCR Correction Results:
  29AACFN8812L124 → 29AACFN8812L1Z4  ✅
  O8AAFCR3201K1ZP → 08AAFCR3201K1ZP  ✅
  33AABCS1429B1Z8 → 33AABCS1429B1Z8  ✅
  27AADCM7832H1ZQ → 27AADCM7832H1ZQ  ✅


## 5️.LLM Data Extraction

In [5]:
import google.generativeai as genai
import json, re, time

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel("gemini-2.5-flash")

EXTRACTION_PROMPT = """
You are a GST invoice data extraction engine for Indian businesses.

Return STRICTLY valid JSON only — no markdown, no backticks, no explanation.

{{
  "vendor_name": "...",
  "gstin": "...",
  "invoice_number": "...",
  "invoice_date": "DD-MM-YYYY",
  "taxable_amount": <number>,
  "cgst": <number>,
  "sgst": <number>,
  "igst": <number>,
  "total_amount": <number>
}}

RULES — read carefully:

vendor_name:
  - The SELLER company name. Usually the TOPMOST bold/all-caps heading.
  - Do NOT extract: subtitles, service descriptions, taglines, project details.
  - Examples of WRONG values: "Software Development & IT Consulting", "Premium Catering & Event Management"
  - Examples of CORRECT values: "NEXATECH SOLUTIONS LLP", "ROYAL RASOI CATERERS"

gstin:
  - SELLER GSTIN only (appears near seller address, NOT Bill To / Consignee).
  - Exactly 15 characters. Return null if not found.
  - Common OCR errors to watch: Z may appear as 2, O may appear as 0.

taxable_amount:
  - Sum of all line items BEFORE tax.
  - Indian format: "4,20,000" = 420000. Remove ALL commas.

total_amount:
  - The GROSS total BEFORE any advance deduction.
  - Look for: GRAND TOTAL / TOTAL AMOUNT / TOTAL INVOICE VALUE / TOTAL AMOUNT PAYABLE.
  - If invoice shows "Balance Payable" after deducting advance, look for the NOTE line
    that says "grand total of Rs. X" — use X, not the balance.
  - Example: Catering invoice shows Balance Payable Rs.3,64,750 but
    note says grand total Rs.4,64,750 → use 4,64,750.

cgst / sgst / igst:
  - Set 0 for any tax type not present.
  - CGST + SGST = intra-state. IGST = inter-state. Never both.

All numeric fields must be plain numbers — NOT strings.

Invoice Text:
{text}
"""

def parse_indian_number(value):
    if value is None: return 0.0
    s = re.sub(r"[^\d.]", "", str(value).replace(",", ""))
    try: return float(s) if s else 0.0
    except ValueError: return 0.0

def extract_invoice_data(text, max_retries=3):
    if not text or len(text.strip()) < 10:
        print("  Warning: Text too short."); return None
    for attempt in range(1, max_retries + 1):
        try:
            response = model.generate_content(EXTRACTION_PROMPT.format(text=text))
            raw = response.text or response.candidates[0].content.parts[0].text
            cleaned = raw.strip()
            if "```" in cleaned:
                for part in cleaned.split("```"):
                    s = part.strip().lstrip("json").strip()
                    if s.startswith("{"): cleaned = s; break
            data = json.loads(cleaned.strip())
            for f in ["taxable_amount","cgst","sgst","igst","total_amount"]:
                data[f] = parse_indian_number(data.get(f))
            return data
        except Exception as e:
            err = str(e)
            if "429" in err or "quota" in err.lower():
                m = re.search(r'seconds.*?(\d+)', err)
                wait = int(m.group(1)) + 5 if m else 15 * attempt
                if attempt < max_retries:
                    print(f"  ⏳ Rate limit — waiting {wait}s (retry {attempt}/{max_retries})...")
                    time.sleep(wait); continue
                else:
                    print("  ✘ Rate limit — all retries exhausted."); return None
            elif isinstance(e, json.JSONDecodeError):
                if attempt < max_retries: time.sleep(3); continue
                return None
            else:
                print(f"  LLM error: {e}"); return None
    return None

C:\Users\PAVAN\anaconda3\envs\MYSpace\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\PAVAN\AppData\Local\Temp\ipykernel_18072\847095356.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 6️.Rule-Based Extraction Layer

In [6]:
import re

def extract_amounts_from_text(text):
    """
    Rule-based extraction of tax amounts directly from invoice text.
    More reliable than LLM for structured numeric fields.
    Returns dict with found values, or empty dict if not found.
    """
    def find_amount(patterns, text):
        """Try each regex pattern, return first match as float."""
        for pat in patterns:
            m = re.search(pat, text, re.IGNORECASE)
            if m:
                raw = m.group(1).replace(',', '')
                try: return float(raw)
                except: pass
        return None

    result = {}

    # Taxable amount
    result['taxable_amount'] = find_amount([
        r'taxable\s+amount[^\d]*([\d,]+\.?\d*)',
        r'taxable[^\d]*([\d,]+\.?\d*)',
    ], text)

    # CGST
    result['cgst'] = find_amount([
        r'cgst\s*@?\s*[\d.]+%?[:\s]*(?:rs\.?\s*)?([\d,]+\.?\d*)',
        r'cgst[:\s]+(?:rs\.?\s*)?([\d,]+\.?\d*)',
    ], text)

    # SGST
    result['sgst'] = find_amount([
        r'sgst\s*@?\s*[\d.]+%?[:\s]*(?:rs\.?\s*)?([\d,]+\.?\d*)',
        r'sgst[:\s]+(?:rs\.?\s*)?([\d,]+\.?\d*)',
    ], text)

    # IGST
    result['igst'] = find_amount([
        r'igst\s*@?\s*[\d.]+%?[:\s]*(?:rs\.?\s*)?([\d,]+\.?\d*)',
        r'igst[:\s]+(?:rs\.?\s*)?([\d,]+\.?\d*)',
    ], text)

    # Total — handle advance/balance case
    # First check for a note line showing gross total (e.g. catering invoice)
    gross_note = re.search(
        r'grand\s+total\s+of\s+(?:rs\.?\s*)?([\d,]+\.?\d*)', text, re.IGNORECASE
    )
    if gross_note:
        result['total_amount'] = float(gross_note.group(1).replace(',', ''))
    else:
        result['total_amount'] = find_amount([
            r'total\s+invoice\s+value[:\s]*(?:rs\.?\s*)?([\d,]+\.?\d*)',
            r'total\s+amount\s+payable[:\s]*(?:rs\.?\s*)?([\d,]+\.?\d*)',
            r'grand\s+total[:\s]*(?:rs\.?\s*)?([\d,]+\.?\d*)',
            r'total\s+amount[:\s]*(?:rs\.?\s*)?([\d,]+\.?\d*)',
        ], text)

    # Remove None values so we can tell what was found
    return {k: v for k, v in result.items() if v is not None}


def merge_rule_llm(rule_data, llm_data):
    """
    Merge rule-based and LLM results.
    Rule values take priority for numeric fields (more reliable).
    LLM values used for text fields and as fallback.
    """
    if not llm_data:
        return llm_data
    merged = dict(llm_data)
    numeric_fields = ['taxable_amount', 'cgst', 'sgst', 'igst', 'total_amount']
    for field in numeric_fields:
        if field in rule_data and rule_data[field] is not None:
            merged[field] = rule_data[field]  # rule wins
    return merged


# ── Quick test ──
# sample = "Taxable Amount: Rs. 52,399.00\nCGST @ 9%: Rs. 4,715.91\nSGST @ 9%: Rs. 4,715.91\nTOTAL AMOUNT: Rs. 61,831.00"
# print(extract_amounts_from_text(sample))

## 7️.Validation + Confidence Score

In [7]:
import re, os, pandas as pd
from datetime import datetime

GSTIN_PATTERN = r"^\d{2}[A-Z]{5}\d{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}$"

def validate_gstin(gstin):
    if not gstin: return None
    g = str(gstin).strip().upper()
    if len(g) != 15: return None
    return g if re.match(GSTIN_PATTERN, g) else None

def validate_date(date_str):
    if not date_str: return None
    for fmt in ("%d-%m-%Y","%d/%m/%Y","%Y-%m-%d","%d %b %Y","%d %B %Y"):
        try: return datetime.strptime(str(date_str).strip(), fmt).strftime("%d-%m-%Y")
        except ValueError: continue
    return date_str

def validate_tax_math(data):
    taxable  = data.get("taxable_amount",0) or 0
    computed = round(taxable+(data.get("cgst",0) or 0)+(data.get("sgst",0) or 0)+(data.get("igst",0) or 0),2)
    actual   = round(data.get("total_amount",0) or 0,2)
    return abs(computed-actual) <= 2.0, computed, actual

def check_duplicate(data, excel_file):
    """v3 fix: uses display column names 'Invoice No.' and 'Vendor Name'."""
    if not os.path.exists(excel_file): return False
    try:
        df = pd.read_excel(excel_file, sheet_name="Invoices", header=2)
        df.columns = df.columns.str.strip()
        if "Vendor Name" not in df.columns or "Invoice No." not in df.columns: return False
        df = df[
            df["Vendor Name"].notna() &
            (df["Vendor Name"].astype(str).str.strip() != "") &
            (df["S.No"].astype(str).str.strip() != "TOTAL")
        ]
        if df.empty: return False
        inv    = str(data.get("invoice_number","")).strip().lower()
        vendor = str(data.get("vendor_name","")).strip().lower()
        for _, row in df.iterrows():
            if (str(row.get("Invoice No.","")).strip().lower() == inv and
                str(row.get("Vendor Name","")).strip().lower() == vendor):
                return True
        return False
    except Exception as e:
        print(f"  Duplicate check error: {e}"); return False

def compute_confidence(data, gstin_was_corrected=False, date_parsed=True, tax_math_ok=True):
    """
    Confidence score 0.0–1.0:
      Start at 1.0, deduct for each issue found.
    """
    score = 1.0
    if not data.get("gstin"):           score -= 0.20  # missing GSTIN
    elif gstin_was_corrected:           score -= 0.10  # OCR correction applied
    if not date_parsed:                 score -= 0.10
    if not tax_math_ok:                 score -= 0.20
    if not data.get("invoice_number"): score -= 0.15
    if not data.get("vendor_name"):    score -= 0.15
    return round(max(score, 0.0), 2)

def validate_data(data, excel_file="gst_data.xlsx"):
    result = {"data":data, "warnings":[], "errors":[], "is_duplicate":False, "confidence":1.0}
    if not data:
        result["errors"].append("No data extracted."); return result

    # [1] GSTIN — apply OCR correction first, then validate
    raw_gstin = data.get("gstin","")
    corrected  = correct_gstin_ocr(raw_gstin) if raw_gstin else raw_gstin
    gstin_was_corrected = (corrected != raw_gstin) and bool(raw_gstin)
    if gstin_was_corrected:
        result["warnings"].append(f"GSTIN OCR corrected: '{raw_gstin}' → '{corrected}'")
    valid_gstin = validate_gstin(corrected)
    if corrected and not valid_gstin:
        result["warnings"].append(f"GSTIN still invalid after correction: '{corrected}' — saved as null.")
    data["gstin"] = valid_gstin

    # [2] Date
    data["invoice_date"] = validate_date(data.get("invoice_date"))
    date_parsed = bool(data["invoice_date"])
    if not date_parsed:
        result["warnings"].append("Invoice date could not be parsed.")

    # [3] Tax math
    ok, computed, actual = validate_tax_math(data)
    if not ok:
        result["warnings"].append(
            f"Tax math mismatch: ₹{computed:,.2f} ≠ ₹{actual:,.2f} "
            f"(diff ₹{abs(computed-actual):.2f}) — verify manually."
        )

    # [4] Required fields
    for f in ["vendor_name","invoice_number","taxable_amount","total_amount"]:
        if not data.get(f):
            result["errors"].append(f"Missing required field: '{f}'")

    # [5] Duplicate
    if check_duplicate(data, excel_file):
        result["is_duplicate"] = True
        result["errors"].append(
            f"DUPLICATE: Invoice '{data.get('invoice_number')}' from "
            f"'{data.get('vendor_name')}' already exists. Skipped."
        )

    # [6] Confidence score
    result["confidence"] = compute_confidence(
        data, gstin_was_corrected, date_parsed, ok
    )
    result["data"] = data
    return result

## 8️.Excel Output

In [8]:
import pandas as pd, os
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime

HEADER_FILL  = PatternFill("solid", start_color="1F4E79")
TOTAL_FILL   = PatternFill("solid", start_color="D6E4F0")
ALT_FILL     = PatternFill("solid", start_color="F2F7FB")
WHITE_FILL   = PatternFill("solid", start_color="FFFFFF")
WARN_FILL    = PatternFill("solid", start_color="FFF3CD")  # amber for low confidence
HEADER_FONT  = Font(bold=True, color="FFFFFF", name="Arial", size=10)
TOTAL_FONT   = Font(bold=True, name="Arial", size=10)
DATA_FONT    = Font(name="Arial", size=9)
THIN         = Side(style="thin", color="BFBFBF")
BORDER       = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
CURRENCY_FMT = '#,##0.00'
PCT_FMT      = '0%'
CENTER       = Alignment(horizontal="center", vertical="center")
LEFT         = Alignment(horizontal="left",   vertical="center")
RIGHT        = Alignment(horizontal="right",  vertical="center")

HEADERS       = ["S.No","Vendor Name","GSTIN","Invoice No.","Invoice Date",
                 "Taxable (₹)","CGST (₹)","SGST (₹)","IGST (₹)","Total (₹)",
                 "Confidence","Processed On"]
WIDTHS        = [6, 28, 20, 18, 14, 16, 12, 12, 12, 16, 11, 18]
CURRENCY_COLS = {6,7,8,9,10}

def _sc(cell, font=None, fill=None, alignment=None, border=None, num_format=None):
    if font:       cell.font          = font
    if fill:       cell.fill          = fill
    if alignment:  cell.alignment     = alignment
    if border:     cell.border        = border
    if num_format: cell.number_format = num_format

def _write_invoices_sheet(ws, df):
    ws.merge_cells("A1:L1")
    ws["A1"].value = "GST Invoice Register"
    _sc(ws["A1"], Font(bold=True,color="1F4E79",name="Arial",size=14),
        PatternFill("solid",start_color="D6E4F0"), CENTER, BORDER)
    ws.row_dimensions[1].height = 28
    ws.merge_cells("A2:L2")
    ws["A2"].value = f"Generated: {datetime.now().strftime('%d %b %Y, %H:%M')}"
    _sc(ws["A2"], Font(italic=True,color="595959",name="Arial",size=9), alignment=CENTER)
    ws.row_dimensions[2].height = 14
    for col,(h,w) in enumerate(zip(HEADERS,WIDTHS),start=1):
        cell = ws.cell(row=3,column=col,value=h)
        _sc(cell, HEADER_FONT, HEADER_FILL, CENTER, BORDER)
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[3].height = 20
    data_start = 4
    for i,(_,row) in enumerate(df.iterrows()):
        r = data_start+i
        conf = row.get("confidence",1.0)
        if isinstance(conf, float) and conf < 0.7:
            fill = WARN_FILL   # amber highlight for low-confidence rows
        else:
            fill = ALT_FILL if i%2==1 else WHITE_FILL
        vals = [i+1, row.get("vendor_name"), row.get("gstin"),
                row.get("invoice_number"), row.get("invoice_date"),
                row.get("taxable_amount"), row.get("cgst"),
                row.get("sgst"), row.get("igst"),
                row.get("total_amount"), row.get("confidence"),
                row.get("processed_on")]
        for col,val in enumerate(vals,start=1):
            cell = ws.cell(row=r,column=col,value=val)
            align = CENTER if col in {1,11} else (RIGHT if col in CURRENCY_COLS else LEFT)
            fmt   = (CURRENCY_FMT if col in CURRENCY_COLS else
                     PCT_FMT if col==11 else None)
            _sc(cell, DATA_FONT, fill, align, BORDER, fmt)
        ws.row_dimensions[r].height = 17
    data_end   = data_start+len(df)-1
    totals_row = data_end+1
    col_map    = {6:"F",7:"G",8:"H",9:"I",10:"J"}
    for col in range(1,13):
        cell = ws.cell(row=totals_row,column=col)
        _sc(cell, TOTAL_FONT, TOTAL_FILL, border=BORDER)
        if col==1: cell.value="TOTAL"; cell.alignment=CENTER
        elif col in col_map:
            L=col_map[col]
            cell.value=f"=SUM({L}{data_start}:{L}{data_end})"
            cell.alignment=RIGHT; cell.number_format=CURRENCY_FMT
        else: cell.alignment=LEFT
    ws.row_dimensions[totals_row].height=20
    ws.freeze_panes="A4"

def _write_summary_sheet(ws, df):
    ws.column_dimensions["A"].width=30; ws.column_dimensions["B"].width=22
    ws.sheet_view.showGridLines=False
    ws.merge_cells("A1:B1")
    ws["A1"].value="GST Summary Report"
    _sc(ws["A1"], Font(bold=True,color="1F4E79",name="Arial",size=13),
        PatternFill("solid",start_color="D6E4F0"), CENTER, BORDER)
    ws.row_dimensions[1].height=26
    rows = [
        ("Total Invoices Processed", len(df)),
        ("Total Taxable Amount",     df["taxable_amount"].sum() if "taxable_amount" in df else 0),
        ("Total CGST",               df["cgst"].sum()           if "cgst" in df else 0),
        ("Total SGST",               df["sgst"].sum()           if "sgst" in df else 0),
        ("Total IGST",               df["igst"].sum()           if "igst" in df else 0),
        ("",""),
        ("FINAL AMOUNT PAYABLE",     df["total_amount"].sum()   if "total_amount" in df else 0),
    ]
    for r_idx,(label,value) in enumerate(rows,start=2):
        a=ws.cell(row=r_idx,column=1,value=label)
        b=ws.cell(row=r_idx,column=2,value=value)
        is_final=label=="FINAL AMOUNT PAYABLE"; is_blank=label==""
        fill=(PatternFill("solid",start_color="1F4E79") if is_final else
              PatternFill("solid",start_color="F2F7FB") if not is_blank else WHITE_FILL)
        fc="FFFFFF" if is_final else "000000"
        for cell in [a,b]:
            _sc(cell, Font(bold=is_final,name="Arial",size=11 if is_final else 10,color=fc),
                fill, LEFT, BORDER if not is_blank else None)
        if isinstance(value,(int,float)) and not is_blank:
            b.number_format=CURRENCY_FMT; b.alignment=RIGHT
        ws.row_dimensions[r_idx].height=22 if is_final else 18

def save_to_excel(record, file_name="gst_data.xlsx"):
    rec = {k: record.get(k) for k in
           ["vendor_name","gstin","invoice_number","invoice_date",
            "taxable_amount","cgst","sgst","igst","total_amount","confidence"]}
    rec["processed_on"] = datetime.now().strftime("%d-%m-%Y %H:%M")
    if os.path.exists(file_name):
        df = pd.read_excel(file_name, sheet_name="Invoices", header=2)
        df.columns = df.columns.str.strip()
        df = df[
            df["Vendor Name"].notna() &
            (df["Vendor Name"].astype(str).str.strip()!="") &
            (df["S.No"].astype(str).str.strip()!="TOTAL")
        ]
        df = df.rename(columns={
            "Vendor Name":"vendor_name","GSTIN":"gstin",
            "Invoice No.":"invoice_number","Invoice Date":"invoice_date",
            "Taxable (₹)":"taxable_amount","CGST (₹)":"cgst",
            "SGST (₹)":"sgst","IGST (₹)":"igst",
            "Total (₹)":"total_amount","Confidence":"confidence",
            "Processed On":"processed_on",
        })
        df = df.drop(columns=["S.No"], errors="ignore")
    else:
        df = pd.DataFrame()
    df = pd.concat([df, pd.DataFrame([rec])], ignore_index=True)
    wb = Workbook()
    ws_inv = wb.active; ws_inv.title = "Invoices"
    _write_invoices_sheet(ws_inv, df)
    _write_summary_sheet(wb.create_sheet("Summary"), df)
    wb.save(file_name)
    print(f"  Saved → {file_name} ({len(df)} invoice(s))")

def add_summary(file_name="gst_data.xlsx"):
    df = pd.read_excel(file_name, sheet_name="Invoices", header=2)
    df.columns = df.columns.str.strip()
    df = df[
        df["Vendor Name"].notna() &
        (df["Vendor Name"].astype(str).str.strip()!="") &
        (df["S.No"].astype(str).str.strip()!="TOTAL")
    ]
    print("\n"+"═"*48)
    print("       GST INVOICE SUMMARY")
    print("═"*48)
    print(f"  Total Invoices    : {len(df)}")
    print(f"  Total Taxable     : ₹{df['Taxable (₹)'].sum():>14,.2f}")
    print(f"  Total CGST        : ₹{df['CGST (₹)'].sum():>14,.2f}")
    print(f"  Total SGST        : ₹{df['SGST (₹)'].sum():>14,.2f}")
    print(f"  Total IGST        : ₹{df['IGST (₹)'].sum():>14,.2f}")
    print("─"*48)
    print(f"  FINAL PAYABLE     : ₹{df['Total (₹)'].sum():>14,.2f}")
    print("═"*48)

## 9️.Single Invoice Pipeline

In [9]:
file_path = "invoice_3_it_services.jpg"  # ← Change to your file (PDF or image)

print(f"Processing: {file_path}")
print("─" * 55)

# Step 1: Extract text
text = extract_text(file_path)
print(f"[1] Extracted {len(text)} characters")

# Step 2: Rule-based extraction (fast, reliable for numeric fields)
rule_data = extract_amounts_from_text(text)
print(f"[2] Rule-based: {rule_data}")

# Step 3: LLM extraction (for text fields + fallback)
llm_data = extract_invoice_data(text)
print(f"[3] LLM output: {llm_data}")

# Step 4: Merge — rule values override LLM for numeric fields
data = merge_rule_llm(rule_data, llm_data)
if data:
    print("[4] Merged:")
    for k,v in data.items(): print(f"      {k:20s}: {v}")

# Step 5: Validate (includes OCR correction + confidence score)
result = validate_data(data, excel_file=EXCEL_FILE)
for w in result["warnings"]: print(f"  ⚠  {w}")
for e in result["errors"]:   print(f"  ✘  {e}")
print(f"  📊 Confidence: {result['confidence']:.0%}")

# Step 6: Save
if not result["errors"] and not result["is_duplicate"]:
    result["data"]["confidence"] = result["confidence"]
    save_to_excel(result["data"], file_name=EXCEL_FILE)
    add_summary(EXCEL_FILE)
elif result["is_duplicate"]:
    print("\n[SKIP] Duplicate — not saved.")
else:
    print("\n[SKIP] Validation failed — not saved.")

Processing: invoice_3_it_services.jpg
───────────────────────────────────────────────────────
[1] Extracted 1245 characters
[2] Rule-based: {'taxable_amount': 974000.0, 'cgst': 87660.0, 'sgst': 87660.0, 'total_amount': 1149320.0}
[3] LLM output: {'vendor_name': 'NEXATECH SOLUTIONS LLP', 'gstin': '29AACFN8812L124', 'invoice_number': 'NXT/2025-26/IT/0219', 'invoice_date': '14-03-2026', 'taxable_amount': 974000.0, 'cgst': 87660.0, 'sgst': 87660.0, 'igst': 0.0, 'total_amount': 1149320.0}
[4] Merged:
      vendor_name         : NEXATECH SOLUTIONS LLP
      gstin               : 29AACFN8812L124
      invoice_number      : NXT/2025-26/IT/0219
      invoice_date        : 14-03-2026
      taxable_amount      : 974000.0
      cgst                : 87660.0
      sgst                : 87660.0
      igst                : 0.0
      total_amount        : 1149320.0
  ⚠  GSTIN OCR corrected: '29AACFN8812L124' → '29AACFN8812L1Z4'
  📊 Confidence: 90%
  Saved → gst_data1.xlsx (1 invoice(s))

═════════════